# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu-yadav-10/Flyrank-ML-starter-template/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane is **Lane 2: Refresh / Content Opportunity Scoring** — deciding which published pages a human review team should look at **first** for refresh, expansion, protection, pruning, or monitoring.

The question *"which ones first?"* makes this a **ranking / scoring** task at the decision level: the product is not a bare yes/no per page but a **priority score per page that sorts into a ranked review queue**. I operationalize it as **binary classification**: estimate P(page is actively losing search visibility | its current signals) and use that predicted probability as the score. The classifier is the engine; the ranked queue is the deliverable. (Clustering would tell me what *kinds* of pages exist — useful later for reason codes — but it cannot say which page to act on first.)

**One-paragraph frame:** For the content-refresh team lead deciding which pages to review first this week, we will build a ranked queue from the anonymized snapshot of 30,000 pages × 44 trailing-90-day signals, scoring each page's probability of active visibility decline, measured by **precision@50** on a client-held-out test split. A wrong call costs reviewer hours spent on pages that never needed work — and, worse, missed declines on pages that did. A plain rule is not enough because decline emerges from many weak signals interacting (no single feature separates decliners above AUC ≈ 0.59 in this slice), with thresholds that shift by content type and position stratum. We will claim **decision-support** results only — never causal claims about how search engines rank pages.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

repo_root = Path.cwd()
if not (repo_root / 'data').is_dir():
    repo_root = Path.cwd().parent.parent

df = pd.read_csv(repo_root / 'data' / 'raw' / 'content_refresh_anonymized.csv')
y = (df['trend_direction'] == 'down')

print('rows:', len(df), '| columns:', df.shape[1])
print('declining pages (score target = 1):', int(y.sum()), f'| share of data: {y.mean():.3f}')
print('other pages (target = 0):', int((~y).sum()))

rows: 30000 | columns: 44
declining pages (score target = 1): 16262 | share of data: 0.542
other pages (target = 0): 13738


## 2. Target or proxy

**Target:** `is_declining_label` = 1 when `trend_direction == 'down'` — impressions fell more than 20% from days 31–60 back (`impressions_prev_30d`) to the most recent 30 days (`impressions_last_30d`). Conceptually it is a **proxy for the thing the team cares about**: *"this page is losing search visibility right now, so reviewing it is timely."*

**Observed or defined? Honestly: both — and I say so.** The ingredients are observed (measured impression counts in two consecutive windows), but the −20% cut is a defined convention, so on this snapshot my label is a **rule-derived proxy built on observed windows**; trained naively, a model partly learns the −20% convention rather than some deeper truth. Two commitments follow: (1) all results are framed as decision-support for *ordering a review queue*, never as ground truth about page quality; (2) in weeks 3+ I rebuild the label on the warehouse daily table as a genuinely forward-looking outcome — features strictly before time T, label measured on impressions in T→T+30 — which removes the definitional circularity.

**Leakage rule that follows immediately:** because `trend_direction` and `trend_pct` are the label's source, they may **never** be features (nor anything derived from them).

In [2]:
sketch = df.loc[df['impressions_prev_30d'] > 0,
                ['content_id', 'impressions_prev_30d', 'impressions_last_30d']].sample(5, random_state=7).copy()
sketch['pct_change_30d'] = ((sketch['impressions_last_30d'] - sketch['impressions_prev_30d'])
                            / sketch['impressions_prev_30d'] * 100).round(1)
sketch['trend_direction'] = df.loc[sketch.index, 'trend_direction']
sketch['is_declining_label'] = (sketch['trend_direction'] == 'down').astype(int)
display(sketch)

labeled_1 = sketch[sketch['is_declining_label'] == 1]
print('sanity check — every sampled label=1 row has pct_change_30d <= -20:',
      bool((labeled_1['pct_change_30d'] <= -20).all()))
print('full-data label counts:', y.value_counts().to_dict())

,content_id,impressions_prev_30d,impressions_last_30d,pct_change_30d,trend_direction,is_declining_label
29865,content_07f879e82acd,149,0,-100.0,down,1
11694,content_fefba358f0c9,4,1,-75.0,down,1
5499,content_eb07d2741db8,6464,4909,-24.1,down,1
4792,content_1cb354dc8c76,8,1,-87.5,down,1
4894,content_5f34072b13af,6280,5949,-5.3,stable,0


sanity check — every sampled label=1 row has pct_change_30d <= -20: True
full-data label counts: {True: 16262, False: 13738}


## 3. Success metric

Named **before** any training: **precision@50** on test clients the model never saw — of the 50 pages my queue puts on top, what fraction truly carry the label. It matches the action exactly: the team works through the *top* of the queue each cycle; a correct pick at position 3,000 has no value, a correct pick at position 12 does.

**What number means "good"? There are two bars, and the higher one is not the baseline:**
- Random ordering lands at the **base rate ≈ 0.542** (54.2% of all pages carry the label) — any ranking must beat *that*, not zero. (A single top-50 draw wobbles ±0.07 around it.)
- The shipped hand-rule baseline scores **0.240** on the reference client-holdout fold — *below* random — because its hard filters (e.g. ≥500 impressions) eject many true decliners from consideration entirely.
- The reference random forest reaches **0.740** (~3× the hand rules), committed in `outputs/model_report.md`.

Working definition of good: **clearly beat the base rate and the hand-rule baseline on held-out clients.** Secondary metrics: recall@K (are we catching the declines we should?) and average precision over the full queue. Precision@K needs only a score column plus the label — provable today, on naive orders, before any model exists:

In [3]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(labels)[order[:k]].mean())

labels_np = y.to_numpy()
rng = np.random.default_rng(42)
naive_orders = {
    'random shuffle':                  rng.random(len(df)),
    'by impressions_90d (desc)':       df['impressions_90d'].to_numpy(),
    'by days_since_last_update (desc)': df['days_since_last_update'].fillna(df['days_since_last_update'].median()).to_numpy(),
}
for name, scores in naive_orders.items():
    print(f'{name:34s} precision@50 = {precision_at_k(scores, labels_np):.3f}')
print(f'{"base rate (random expectation)":34s} {y.mean():.3f}')

random shuffle                     precision@50 = 0.440
by impressions_90d (desc)          precision@50 = 0.420
by days_since_last_update (desc)   precision@50 = 0.560
base rate (random expectation)     0.542


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (page) for one client, summarized over a trailing 90-day window** ending at export time — with two nested 30-day sub-windows that feed the trend label. It is *not* one row per client (there are only 32 of those), not one row per day (that is the warehouse fact table's grain), and not one row per query.

The grain check below proves it: 30,000 rows, 30,000 distinct `content_id`s, 32 clients — pages per client range from 3 to 7,008 (median 567). That imbalance is also why every later train/test split must hold out whole **clients**, never individual rows.

Two honesty flags already visible in this slice: `avg_position = 0` means *"no position data"* (1,205 rows), not a great rank; and missingness follows content type (7,699 blank `word_count`, 2,468 blank `search_volume`), so `has_*` flag columns beat a blind `fillna(0)` that would silently encode content type into the features.

In [4]:
unit = df[[
    'content_id', 'client_id', 'content_type',
    'impressions_90d', 'clicks_90d', 'ctr', 'avg_position',
    'engagement_rate', 'days_since_last_update', 'search_volume',
    'impressions_prev_30d', 'impressions_last_30d', 'trend_direction',
]].copy()
unit['is_declining_label'] = (unit['trend_direction'] == 'down').astype(int)
unit = unit.drop(columns='trend_direction')

print('shape:', unit.shape)
print('grain check — unique content_id:', unit['content_id'].nunique(),
      '| duplicated ids:', int(unit['content_id'].duplicated().sum()),
      '| clients:', unit['client_id'].nunique())
per_client = df.groupby('client_id').size()
print('pages per client — min:', per_client.min(), '| median:', int(per_client.median()), '| max:', per_client.max())
print('missingness flags — word_count NaN:', int(df['word_count'].isna().sum()),
      '| search_volume NaN:', int(df['search_volume'].isna().sum()),
      '| avg_position == 0 (no data):', int((df['avg_position'] == 0).sum()))
display(unit.head(8))

shape: (30000, 13)
grain check — unique content_id: 30000 | duplicated ids: 0 | clients: 32
pages per client — min: 3 | median: 567 | max: 7008
missingness flags — word_count NaN: 7699 | search_volume NaN: 2468 | avg_position == 0 (no data): 1205


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,days_since_last_update,search_volume,impressions_prev_30d,impressions_last_30d,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,5.88,20,10.0,987,578,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,0.00,25,90.0,5915,2501,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,0.00,20,0.0,6089,2382,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,1.28,22,10.0,4206,3626,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,0.00,14,0.0,6452,4211,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,0.03,8.5,0.00,20,720.0,1009,617,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,0.00,7.0,0.00,20,0.0,13,1,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,0.06,21.2,3.57,22,590.0,632,636,0


## 5. Why ML beats a fixed rule here

The fixed rule already exists in this repo — the baseline's hand-written thresholds like `stale_visible_page` ("old AND visible") or `thin_visible_page`. Three pieces of evidence, all reproducible in the cell below, say that is not enough here:

1. **Every single signal is weak alone.** The best single-feature AUC against the label is ≈ 0.59 (`content_age_days`); CTR, engagement rate, and average position sit at ≈ 0.50–0.53. No single `if impressions > …` line can isolate these pages.
2. **The pattern is non-monotonic and interactive.** Among keyword articles, decline *peaks* in the `striking` and `page_1` tiers (~0.58–0.62) and *drops* to 0.37 in `top_3`; feedly articles in `top_3` almost never decline (0.04). Expressing content-type × position × age × volume interactions as stacked thresholds explodes combinatorially — and any hand-tuned cutoff is brittle as the mix shifts.
3. **It measurably pays.** On the same client-held-out fold, the reference pipeline's hand rules reach precision@50 = 0.240 while a random forest reaches 0.740 — roughly 3× more genuine finds per week out of the same fixed review capacity.

**Honest limits:** rules do not die — they become the **reason codes** attached to each queue entry ("why am I being shown this page?") and the fallback when the model is silent. ML earns the *ordering*; rules keep it explainable. And none of this claims causation about search rankings — the claims are directional and decision-support only.

In [5]:
from sklearn.metrics import roc_auc_score

single_signal_auc = {}
for f in ['content_age_days', 'ctr', 'avg_position', 'engagement_rate',
          'impressions_90d', 'search_volume', 'days_since_last_update']:
    s = df[f].fillna(df[f].median())
    auc = roc_auc_score(y, s)
    single_signal_auc[f] = round(max(auc, 1 - auc), 3)
print('single-feature AUC vs declining label:')
print(pd.Series(single_signal_auc).sort_values(ascending=False).to_string())

print()
combo = (df.assign(is_down=y.astype(int))
           .groupby(['content_type', 'position_tier'])['is_down']
           .agg(pages='size', declining_rate='mean')
           .query('pages >= 100'))
print('declining rate by content_type x position_tier (strata with >= 100 pages):')
print(combo.round(3).to_string())

single-feature AUC vs declining label:
content_age_days          0.591
impressions_90d           0.584
search_volume             0.560
avg_position              0.530
days_since_last_update    0.527
ctr                       0.515
engagement_rate           0.504

declining rate by content_type x position_tier (strata with >= 100 pages):
                                  pages  declining_rate
content_type       position_tier                       
comparison article page_1           435           0.570
                   striking         180           0.506
feedly article     page_1           878           0.470
                   striking         181           0.519
                   top_3            923           0.040
keyword article    deep            1304           0.341
                   page_1         10501           0.578
                   page_3_5        7064           0.561
                   striking        6943           0.615
                   top_3           1395      

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.